In [0]:
pip install -q beautifulsoup4 

In [0]:
dbutils.library.restartPython()

In [0]:
import requests
import re
import json

from bs4 import BeautifulSoup
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

from pyspark.sql.functions import col, lit

In [0]:
job_portals = spark.table("main_catalogue.jobs.workday_portals")

In [0]:
# ------------------------------------------------------------------
# Sessions
# ------------------------------------------------------------------

workday_session = requests.Session()
scrape_session = requests.Session()

# ------------------------------------------------------------------
# Workday API
# ------------------------------------------------------------------

def get_job_posting(api_url, facet, locations, key):

    payload = {
        "appliedFacets": {
            facet: locations
        },
        "limit": 20,
        "offset": 0,
        "searchText": key
    }

    response = workday_session.post(
        api_url,
        json=payload,
        timeout=20
    )

    response.raise_for_status()

    return response.json().get("jobPostings", [])


# ------------------------------------------------------------------
# Date helper
# ------------------------------------------------------------------

def get_days(postedOn):

    if not postedOn:
        return None

    if "Today" in postedOn:
        return 0

    if "Yesterday" in postedOn:
        return 1

    match = re.search(r"(\d+)", postedOn)

    return int(match.group(1)) if match else None


# ------------------------------------------------------------------
# Search keywords
# ------------------------------------------------------------------

search_keys = [
    "Azure",
    "Data Engineer",
    "Databricks",
    "Big Data",
    "ETL",
    "Data Platform",
    "Pyspark",
    "SQL"
]


# ------------------------------------------------------------------
# Process company + keyword
# ------------------------------------------------------------------

def process_request(row, key):

    api_url = row["api_url"]

    try:

        hostname = urlparse(api_url).hostname

        company = (
            hostname
            .split(".myworkdayjobs")[0]
            .split(".")[0]
        )

        base_url = row["url"]

        job_postings = get_job_posting(
            api_url,
            row["facet_location"],
            row["locations"],
            key
        )

        result = []

        for job in job_postings:

            result.append({
                "title": job.get("title"),
                "externalPath": job.get("externalPath"),
                "locationsText": job.get("locationsText"),
                "postedOn": job.get("postedOn"),
                "company": company,
                "keyword": key,
                "url": base_url.rstrip("/") + job.get("externalPath", ""),
                "api_url": api_url[:-5].rstrip("/") + job.get("externalPath", ""),
                "days": get_days(job.get("postedOn"))
            })

        return result

    except Exception as e:

        print(
            f"Error | URL={row['url']} | KEY={key} | {str(e)[:200]}"
        )

        return []


# ------------------------------------------------------------------
# Fetch jobs in parallel
# ------------------------------------------------------------------

jobs = []

with ThreadPoolExecutor(max_workers=50) as executor:

    futures = []

    for row in job_portals.toLocalIterator():

        for key in search_keys:

            futures.append(
                executor.submit(
                    process_request,
                    row,
                    key
                )
            )

    total = len(futures)

    for idx, future in enumerate(as_completed(futures), start=1):

        if idx % 100 == 0:
            print(f"Completed {idx}/{total}")

        jobs.extend(future.result())


# ------------------------------------------------------------------
# Create dataframe
# ------------------------------------------------------------------

jobs_df = spark.createDataFrame(jobs)

jobs_df = jobs_df.dropDuplicates(
    ["company", "externalPath"]
)

display(jobs_df)

# ------------------------------------------------------------------
# Today's jobs
# ------------------------------------------------------------------

todays_jobs = (
    jobs_df
    .filter(col("days") == 0)
    .dropDuplicates(["url"])
    .withColumn("hrs_ago", lit("Today"))
    .withColumnRenamed("locationsText", "location")
)

todays_jobs.createOrReplaceTempView("todays_jobs")

# ------------------------------------------------------------------
# Remove already processed jobs
# ------------------------------------------------------------------

new_jobs = spark.sql("""
select a.*
from todays_jobs a
left anti join main_catalogue.jobs.workday b
on a.url = b.url
""")

print(f"New jobs count: {new_jobs.count()}")

# ------------------------------------------------------------------
# Description Scraper
# ------------------------------------------------------------------

def get_description(url):

    try:

        response = scrape_session.get(
            url,
            timeout=20,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        script_tag = soup.find(
            "script",
            attrs={"type": "application/ld+json"}
        )

        if not script_tag:
            return None

        data = json.loads(script_tag.text)

        if isinstance(data, dict):
            return data.get("description")

        if isinstance(data, list):
            for item in data:
                if isinstance(item, dict) and item.get("description"):
                    return item["description"]

        return None

    except Exception:
        return None


# ------------------------------------------------------------------
# Scrape descriptions in parallel
# ------------------------------------------------------------------

rows = new_jobs.collect()

def enrich_row(row):

    r = row.asDict()

    r["description"] = get_description(
        row["url"]
    )

    return r


results = []

with ThreadPoolExecutor(max_workers=30) as executor:

    futures = [
        executor.submit(enrich_row, row)
        for row in rows
    ]

    total = len(futures)

    for idx, future in enumerate(as_completed(futures), start=1):

        if idx % 50 == 0:
            print(f"Descriptions completed {idx}/{total}")

        results.append(future.result())


# ------------------------------------------------------------------
# Final dataframe
# ------------------------------------------------------------------

result_df = spark.createDataFrame(results)

display(result_df)

In [0]:
result_df.select("url","title","company","location","description").write.mode("overwrite").option("overwriteSchema", True).saveAsTable("staging_catalogue.jobs.workday")